In [1]:
%pip install pyspark pandas matplotlib seaborn scikit-learn requests

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ChicagoCrimeBigData")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config(
        "spark.hadoop.fs.file.impl",
        "org.apache.hadoop.fs.RawLocalFileSystem"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print(
    "Hadoop version:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Spark version: 4.2.0
Hadoop version: 3.5.0


In [3]:
#define the data paths

import os

BASE_PATH = r"C:\Users\HP\Documents\Big Data Technology"

CRIME_PATH = os.path.join(
    BASE_PATH,
    "data",
    "raw",
    "Crimes.csv"
)

SOCIO_PATH = os.path.join(
    BASE_PATH,
    "data",
    "raw",
    "Socioeconomic.csv"
)

WEATHER_PATH = os.path.join(
    BASE_PATH,
    "data",
    "raw",
    "Chicago_weather.csv"
)

print("Crime:", CRIME_PATH)
print("Socio-economic:", SOCIO_PATH)
print("Weather:", WEATHER_PATH)

Crime: C:\Users\HP\Documents\Big Data Technology\data\raw\Crimes.csv
Socio-economic: C:\Users\HP\Documents\Big Data Technology\data\raw\Socioeconomic.csv
Weather: C:\Users\HP\Documents\Big Data Technology\data\raw\Chicago_weather.csv


In [4]:
#verify the files

print("Crime exists:", os.path.exists(CRIME_PATH))
print("Socio-economic exists:", os.path.exists(SOCIO_PATH))
print("Weather exists:", os.path.exists(WEATHER_PATH))

Crime exists: True
Socio-economic exists: True
Weather exists: True


In [5]:
#Ingest the Crime Dataset

crime_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CRIME_PATH)
)

print("Crime rows:", crime_raw.count())
print("Crime columns:", len(crime_raw.columns))

Crime rows: 7784664
Crime columns: 22


In [6]:
crime_raw.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [7]:
crime_raw.show(5, truncate=False)

+--------+-----------+----------------------+---------------------+----+------------+-----------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|ID      |Case Number|Date                  |Block                |IUCR|Primary Type|Description            |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location                     |
+--------+-----------+----------------------+---------------------+----+------------+-----------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|10224738|HY411648   |09/05/2015 01:30:00 PM|043XX S WOOD ST      |0486|BATTERY     |DOMESTIC BAT

In [8]:
#Examine the Crime Columns

print(crime_raw.columns)

['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


In [9]:
#Examine Crime Data Quality

crime_raw.select(
    F.count("*").alias("total_rows"),
    F.count("Date").alias("valid_date"),
    F.count("Latitude").alias("valid_latitude"),
    F.count("Longitude").alias("valid_longitude"),
    F.count("District").alias("valid_district"),
    F.count("Community Area").alias("valid_community_area")
).show()

+----------+----------+--------------+---------------+--------------+--------------------+
|total_rows|valid_date|valid_latitude|valid_longitude|valid_district|valid_community_area|
+----------+----------+--------------+---------------+--------------+--------------------+
|   7784664|   7784664|       7697816|        7697816|       7784617|             7171188|
+----------+----------+--------------+---------------+--------------+--------------------+



In [10]:
 #Define the Explicit Crime Schema

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    DoubleType, BooleanType
)

crime_schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Case Number", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Block", StringType(), True),
    StructField("IUCR", StringType(), True),
    StructField("Primary Type", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Location Description", StringType(), True),
    StructField("Arrest", BooleanType(), True),
    StructField("Domestic", BooleanType(), True),
    StructField("Beat", IntegerType(), True),
    StructField("District", IntegerType(), True),
    StructField("Ward", IntegerType(), True),
    StructField("Community Area", IntegerType(), True),
    StructField("FBI Code", StringType(), True),
    StructField("X Coordinate", DoubleType(), True),
    StructField("Y Coordinate", DoubleType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Updated On", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True)
])

In [11]:
#Reload Crime Using the Enforced Schema

crime = (
    spark.read
    .option("header", True)
    .schema(crime_schema)
    .csv(CRIME_PATH)
)

print("Crime rows:", crime.count())
print("Crime columns:", len(crime.columns))

Crime rows: 7784664
Crime columns: 22


In [12]:
crime.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [13]:
#Convert the Crime Timestamp

from pyspark.sql import functions as F

crime = crime.withColumn(
    "crime_timestamp",
    F.to_timestamp("Date", "MM/dd/yyyy hh:mm:ss a")
)

crime.select(
    "Date",
    "crime_timestamp"
).show(10, truncate=False)

+----------------------+-------------------+
|Date                  |crime_timestamp    |
+----------------------+-------------------+
|09/05/2015 01:30:00 PM|2015-09-05 13:30:00|
|09/04/2015 11:30:00 AM|2015-09-04 11:30:00|
|09/01/2018 12:01:00 AM|2018-09-01 00:01:00|
|09/05/2015 12:45:00 PM|2015-09-05 12:45:00|
|09/05/2015 01:00:00 PM|2015-09-05 13:00:00|
|09/05/2015 10:55:00 AM|2015-09-05 10:55:00|
|09/04/2015 06:00:00 PM|2015-09-04 18:00:00|
|09/05/2015 01:00:00 PM|2015-09-05 13:00:00|
|09/05/2015 11:30:00 AM|2015-09-05 11:30:00|
|05/01/2016 12:25:00 AM|2016-05-01 00:25:00|
+----------------------+-------------------+
only showing top 10 rows


In [14]:
#Create a Clean Crime Date

crime = crime.withColumn(
    "crime_date",
    F.to_date("crime_timestamp")
)

crime.select(
    "crime_timestamp",
    "crime_date"
).show(10)

+-------------------+----------+
|    crime_timestamp|crime_date|
+-------------------+----------+
|2015-09-05 13:30:00|2015-09-05|
|2015-09-04 11:30:00|2015-09-04|
|2018-09-01 00:01:00|2018-09-01|
|2015-09-05 12:45:00|2015-09-05|
|2015-09-05 13:00:00|2015-09-05|
|2015-09-05 10:55:00|2015-09-05|
|2015-09-04 18:00:00|2015-09-04|
|2015-09-05 13:00:00|2015-09-05|
|2015-09-05 11:30:00|2015-09-05|
|2016-05-01 00:25:00|2016-05-01|
+-------------------+----------+
only showing top 10 rows


In [15]:
#Create Temporal Features

crime = (
    crime
    .withColumn("crime_year", F.year("crime_timestamp"))
    .withColumn("crime_month", F.month("crime_timestamp"))
    .withColumn("crime_day", F.dayofmonth("crime_timestamp"))
    .withColumn("crime_hour", F.hour("crime_timestamp"))
    .withColumn("crime_day_of_week", F.dayofweek("crime_timestamp"))
)

crime.select(
    "crime_timestamp",
    "crime_year",
    "crime_month",
    "crime_day",
    "crime_hour",
    "crime_day_of_week"
).show(10)

+-------------------+----------+-----------+---------+----------+-----------------+
|    crime_timestamp|crime_year|crime_month|crime_day|crime_hour|crime_day_of_week|
+-------------------+----------+-----------+---------+----------+-----------------+
|2015-09-05 13:30:00|      2015|          9|        5|        13|                7|
|2015-09-04 11:30:00|      2015|          9|        4|        11|                6|
|2018-09-01 00:01:00|      2018|          9|        1|         0|                7|
|2015-09-05 12:45:00|      2015|          9|        5|        12|                7|
|2015-09-05 13:00:00|      2015|          9|        5|        13|                7|
|2015-09-05 10:55:00|      2015|          9|        5|        10|                7|
|2015-09-04 18:00:00|      2015|          9|        4|        18|                6|
|2015-09-05 13:00:00|      2015|          9|        5|        13|                7|
|2015-09-05 11:30:00|      2015|          9|        5|        11|           

In [16]:
#Clean District

crime = crime.withColumn(
    "district_clean",
    F.when(
        (F.col("District").isNotNull()) &
        (F.col("District") > 0),
        F.col("District")
    ).otherwise(None)
)

crime.select(
    "District",
    "district_clean"
).show(10)

+--------+--------------+
|District|district_clean|
+--------+--------------+
|       9|             9|
|      15|            15|
|       6|             6|
|      14|            14|
|      15|            15|
|       6|             6|
|      14|            14|
|      10|            10|
|      12|            12|
|       8|             8|
+--------+--------------+
only showing top 10 rows


In [17]:
#Clean Community Area

crime = crime.withColumn(
    "community_area_clean",
    F.when(
        (F.col("Community Area").isNotNull()) &
        (F.col("Community Area") > 0),
        F.col("Community Area")
    ).otherwise(None)
)

crime.select(
    "Community Area",
    "community_area_clean"
).show(10)

+--------------+--------------------+
|Community Area|community_area_clean|
+--------------+--------------------+
|            61|                  61|
|            25|                  25|
|            44|                  44|
|            21|                  21|
|            25|                  25|
|            71|                  71|
|            24|                  24|
|            31|                  31|
|            27|                  27|
|            63|                  63|
+--------------+--------------------+
only showing top 10 rows


In [18]:
#Clean Latitude and Longitude

crime = (
    crime
    .withColumn(
        "latitude_clean",
        F.when(
            (F.col("Latitude") >= 41.0) &
            (F.col("Latitude") <= 43.0),
            F.col("Latitude")
        ).otherwise(None)
    )
    .withColumn(
        "longitude_clean",
        F.when(
            (F.col("Longitude") >= -88.0) &
            (F.col("Longitude") <= -87.0),
            F.col("Longitude")
        ).otherwise(None)
    )
)

In [19]:
crime.select(
    "Latitude",
    "latitude_clean",
    "Longitude",
    "longitude_clean"
).show(10)

+------------+--------------+-------------+---------------+
|    Latitude|latitude_clean|    Longitude|longitude_clean|
+------------+--------------+-------------+---------------+
|41.815117282|  41.815117282|-87.669999562|  -87.669999562|
|41.895080471|  41.895080471|-87.765400451|  -87.765400451|
|        NULL|          NULL|         NULL|           NULL|
|41.937405765|  41.937405765|-87.716649687|  -87.716649687|
|41.881903443|  41.881903443|-87.755121152|  -87.755121152|
|41.744378879|  41.744378879|-87.658430635|  -87.658430635|
|41.914635603|  41.914635603|-87.681630909|  -87.681630909|
|41.851988885|  41.851988885|-87.689219118|  -87.689219118|
| 41.88281374|   41.88281374|-87.704325717|  -87.704325717|
|        NULL|          NULL|         NULL|           NULL|
+------------+--------------+-------------+---------------+
only showing top 10 rows


In [20]:
#Check the Cleaning Results

crime.select(
    F.count("*").alias("total_rows"),

    F.sum(
        F.col("crime_timestamp").isNull().cast("int")
    ).alias("invalid_timestamps"),

    F.sum(
        F.col("district_clean").isNull().cast("int")
    ).alias("missing_district"),

    F.sum(
        F.col("community_area_clean").isNull().cast("int")
    ).alias("missing_community_area"),

    F.sum(
        F.col("latitude_clean").isNull().cast("int")
    ).alias("invalid_latitude"),

    F.sum(
        F.col("longitude_clean").isNull().cast("int")
    ).alias("invalid_longitude")
).show()

+----------+------------------+----------------+----------------------+----------------+-----------------+
|total_rows|invalid_timestamps|missing_district|missing_community_area|invalid_latitude|invalid_longitude|
+----------+------------------+----------------+----------------------+----------------+-----------------+
|   7784664|                 0|              47|                613552|           86966|            86966|
+----------+------------------+----------------+----------------------+----------------+-----------------+



In [21]:
#Create the Final Clean Crime Dataset

crime_clean = crime.filter(
    F.col("crime_timestamp").isNotNull()
)

In [22]:
#Remove Exact Duplicate Records

duplicate_count = (
    crime_clean.count()
    - crime_clean.dropDuplicates(["ID"]).count()
)

print("Potential duplicate records:", duplicate_count)

Potential duplicate records: 0


In [23]:
#Inspect the Final Crime Schema
crime_clean.printSchema()

crime_clean.select(
    "ID",
    "Primary Type",
    "Arrest",
    "crime_timestamp",
    "crime_date",
    "crime_year",
    "crime_month",
    "crime_hour",
    "district_clean",
    "community_area_clean",
    "latitude_clean",
    "longitude_clean"
).show(10, truncate=False)

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- crime_timestamp: timestamp (nullable = true)
 |-- crime_date: date (nullable = true)
 |-- crim

In [24]:
#Cache the Clean Dataset

crime_clean.cache()

print("Cached crime records:", crime_clean.count())

Cached crime records: 7784664


In [25]:
#Inspect the Socio-Economic Dataset

# Inspect socio-economic dataset

socioeconomic_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOCIO_PATH)
)

print("Rows:", socioeconomic_raw.count())
print("Columns:", len(socioeconomic_raw.columns))

socioeconomic_raw.printSchema()

Rows: 78
Columns: 9
root
 |-- Community Area Number: integer (nullable = true)
 |-- COMMUNITY AREA NAME: string (nullable = true)
 |-- PERCENT OF HOUSING CROWDED: double (nullable = true)
 |-- PERCENT HOUSEHOLDS BELOW POVERTY: double (nullable = true)
 |-- PERCENT AGED 16+ UNEMPLOYED: double (nullable = true)
 |-- PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA: double (nullable = true)
 |-- PERCENT AGED UNDER 18 OR OVER 64: double (nullable = true)
 |-- PER CAPITA INCOME : integer (nullable = true)
 |-- HARDSHIP INDEX: integer (nullable = true)



In [26]:
socioeconomic_raw.show(10, truncate=False)

+---------------------+-------------------+--------------------------+--------------------------------+---------------------------+--------------------------------------------+--------------------------------+------------------+--------------+
|Community Area Number|COMMUNITY AREA NAME|PERCENT OF HOUSING CROWDED|PERCENT HOUSEHOLDS BELOW POVERTY|PERCENT AGED 16+ UNEMPLOYED|PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA|PERCENT AGED UNDER 18 OR OVER 64|PER CAPITA INCOME |HARDSHIP INDEX|
+---------------------+-------------------+--------------------------+--------------------------------+---------------------------+--------------------------------------------+--------------------------------+------------------+--------------+
|1                    |Rogers Park        |7.7                       |23.6                            |8.7                        |18.2                                        |27.5                            |23939             |39            |
|2                    |W

In [27]:
print(socioeconomic_raw.columns)

['Community Area Number', 'COMMUNITY AREA NAME', 'PERCENT OF HOUSING CROWDED', 'PERCENT HOUSEHOLDS BELOW POVERTY', 'PERCENT AGED 16+ UNEMPLOYED', 'PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA', 'PERCENT AGED UNDER 18 OR OVER 64', 'PER CAPITA INCOME ', 'HARDSHIP INDEX']


In [28]:
#Check Socio-Economic Data Quality

# Step 22: Check missing values

socioeconomic_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in socioeconomic_raw.columns
]).show()

print(
    "Duplicate rows:",
    socioeconomic_raw.count() - socioeconomic_raw.dropDuplicates().count()
)

+---------------------+-------------------+--------------------------+--------------------------------+---------------------------+--------------------------------------------+--------------------------------+------------------+--------------+
|Community Area Number|COMMUNITY AREA NAME|PERCENT OF HOUSING CROWDED|PERCENT HOUSEHOLDS BELOW POVERTY|PERCENT AGED 16+ UNEMPLOYED|PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA|PERCENT AGED UNDER 18 OR OVER 64|PER CAPITA INCOME |HARDSHIP INDEX|
+---------------------+-------------------+--------------------------+--------------------------------+---------------------------+--------------------------------------------+--------------------------------+------------------+--------------+
|                    1|                  0|                         0|                               0|                          0|                                           0|                               0|                 0|             1|
+---------------------+-

In [29]:
#Check the Community Area Values

socioeconomic_raw.select(
    "Community Area Number"
).distinct().orderBy("Community Area Number").show(100)

+---------------------+
|Community Area Number|
+---------------------+
|                 NULL|
|                    1|
|                    2|
|                    3|
|                    4|
|                    5|
|                    6|
|                    7|
|                    8|
|                    9|
|                   10|
|                   11|
|                   12|
|                   13|
|                   14|
|                   15|
|                   16|
|                   17|
|                   18|
|                   19|
|                   20|
|                   21|
|                   22|
|                   23|
|                   24|
|                   25|
|                   26|
|                   27|
|                   28|
|                   29|
|                   30|
|                   31|
|                   32|
|                   33|
|                   34|
|                   35|
|                   36|
|                   37|
|               

In [30]:
#Standardize Socio-Economic Columns
# Standardize socio-economic column names

socioeconomic = (
    socioeconomic_raw
    .withColumnRenamed("Community Area Number", "community_area")
    .withColumnRenamed("COMMUNITY AREA NAME", "community_area_name")
    .withColumnRenamed("PERCENT OF HOUSING CROWDED", "housing_crowded_pct")
    .withColumnRenamed("PERCENT HOUSEHOLDS BELOW POVERTY", "poverty_pct")
    .withColumnRenamed("PERCENT AGED 16+ UNEMPLOYED", "unemployment_pct")
    .withColumnRenamed(
        "PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA",
        "no_high_school_pct"
    )
    .withColumnRenamed(
        "PERCENT AGED UNDER 18 OR OVER 64",
        "dependent_age_pct"
    )
    .withColumnRenamed("PER CAPITA INCOME ", "per_capita_income")
    .withColumnRenamed("HARDSHIP INDEX", "hardship_index")
)

socioeconomic.printSchema()

root
 |-- community_area: integer (nullable = true)
 |-- community_area_name: string (nullable = true)
 |-- housing_crowded_pct: double (nullable = true)
 |-- poverty_pct: double (nullable = true)
 |-- unemployment_pct: double (nullable = true)
 |-- no_high_school_pct: double (nullable = true)
 |-- dependent_age_pct: double (nullable = true)
 |-- per_capita_income: integer (nullable = true)
 |-- hardship_index: integer (nullable = true)



In [31]:
#Enforce the Socio-Economic Schema
# Enforce data types

socioeconomic = (
    socioeconomic
    .withColumn(
        "community_area",
        F.col("community_area").cast("integer")
    )
    .withColumn(
        "housing_crowded_pct",
        F.col("housing_crowded_pct").cast("double")
    )
    .withColumn(
        "poverty_pct",
        F.col("poverty_pct").cast("double")
    )
    .withColumn(
        "unemployment_pct",
        F.col("unemployment_pct").cast("double")
    )
    .withColumn(
        "no_high_school_pct",
        F.col("no_high_school_pct").cast("double")
    )
    .withColumn(
        "dependent_age_pct",
        F.col("dependent_age_pct").cast("double")
    )
    .withColumn(
        "per_capita_income",
        F.col("per_capita_income").cast("double")
    )
    .withColumn(
        "hardship_index",
        F.col("hardship_index").cast("double")
    )
)

In [32]:
#Check Socio-Economic Data
socioeconomic.show(10, truncate=False)
print("Socio-economic rows:", socioeconomic.count())
print("Socio-economic columns:", len(socioeconomic.columns))

+--------------+-------------------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+
|community_area|community_area_name|housing_crowded_pct|poverty_pct|unemployment_pct|no_high_school_pct|dependent_age_pct|per_capita_income|hardship_index|
+--------------+-------------------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+
|1             |Rogers Park        |7.7                |23.6       |8.7             |18.2              |27.5             |23939.0          |39.0          |
|2             |West Ridge         |7.8                |17.2       |8.8             |20.8              |38.5             |23040.0          |46.0          |
|3             |Uptown             |3.8                |24.0       |8.9             |11.8              |22.2             |35787.0          |20.0          |
|4             |Lincoln Square     |3.4                |10.9    

In [33]:
#Check Missing Values
socioeconomic.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in socioeconomic.columns
]).show()

+--------------+-------------------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+
|community_area|community_area_name|housing_crowded_pct|poverty_pct|unemployment_pct|no_high_school_pct|dependent_age_pct|per_capita_income|hardship_index|
+--------------+-------------------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+
|             1|                  0|                  0|          0|               0|                 0|                0|                0|             1|
+--------------+-------------------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+



In [34]:
#Check Duplicate Community Areas
print(
    "Total rows:",
    socioeconomic.count()
)

print(
    "Unique community areas:",
    socioeconomic.select("community_area").distinct().count()
)

socioeconomic.groupBy("community_area").count().orderBy("community_area").show(100)

Total rows: 78
Unique community areas: 78
+--------------+-----+
|community_area|count|
+--------------+-----+
|          NULL|    1|
|             1|    1|
|             2|    1|
|             3|    1|
|             4|    1|
|             5|    1|
|             6|    1|
|             7|    1|
|             8|    1|
|             9|    1|
|            10|    1|
|            11|    1|
|            12|    1|
|            13|    1|
|            14|    1|
|            15|    1|
|            16|    1|
|            17|    1|
|            18|    1|
|            19|    1|
|            20|    1|
|            21|    1|
|            22|    1|
|            23|    1|
|            24|    1|
|            25|    1|
|            26|    1|
|            27|    1|
|            28|    1|
|            29|    1|
|            30|    1|
|            31|    1|
|            32|    1|
|            33|    1|
|            34|    1|
|            35|    1|
|            36|    1|
|            37|    1|
|            38

In [35]:
#Check Invalid Community Area Values

socioeconomic.select(
    F.min("community_area").alias("min_community_area"),
    F.max("community_area").alias("max_community_area")
).show()

socioeconomic_clean = socioeconomic.filter(
    (F.col("community_area").isNotNull()) &
    (F.col("community_area") > 0)
)

print(
    "Clean socio-economic rows:",
    socioeconomic_clean.count()
)

+------------------+------------------+
|min_community_area|max_community_area|
+------------------+------------------+
|                 1|                77|
+------------------+------------------+

Clean socio-economic rows: 77


In [36]:
#Check the Final Socio-Economic Dataset
socioeconomic_clean.printSchema()
socioeconomic_clean.show(10, truncate=False)

root
 |-- community_area: integer (nullable = true)
 |-- community_area_name: string (nullable = true)
 |-- housing_crowded_pct: double (nullable = true)
 |-- poverty_pct: double (nullable = true)
 |-- unemployment_pct: double (nullable = true)
 |-- no_high_school_pct: double (nullable = true)
 |-- dependent_age_pct: double (nullable = true)
 |-- per_capita_income: double (nullable = true)
 |-- hardship_index: double (nullable = true)

+--------------+-------------------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+
|community_area|community_area_name|housing_crowded_pct|poverty_pct|unemployment_pct|no_high_school_pct|dependent_age_pct|per_capita_income|hardship_index|
+--------------+-------------------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+
|1             |Rogers Park        |7.7                |23.6       |8.7             |18.2   

In [37]:
#Validate the Crime/Socio-Economic Join
crime_community_match = (
    crime_clean
    .join(
        socioeconomic_clean,
        crime_clean.community_area_clean ==
        socioeconomic_clean.community_area,
        "left"
    )
)

print("Crime records:", crime_clean.count())
print("Joined records:", crime_community_match.count())

unmatched = crime_community_match.filter(
    F.col("community_area").isNull()
).count()

print("Crime records without socio-economic match:", unmatched)

total_crime = crime_clean.count()

matched = total_crime - unmatched

print(
    "Socio-economic match percentage:",
    round((matched / total_crime) * 100, 2),
    "%"
)

Crime records: 7784664
Joined records: 7784664
Crime records without socio-economic match: 613552
Socio-economic match percentage: 92.12 %


In [38]:
#Load the Weather Dataset

weather_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(WEATHER_PATH)
)

print("Weather rows:", weather_raw.count())
print("Weather columns:", len(weather_raw.columns))

Weather rows: 10728
Weather columns: 56


In [39]:
weather_raw.printSchema()
weather_raw.show(10, truncate=False)
print(weather_raw.columns)

root
 |-- STATION: string (nullable = true)
 |-- DATE: date (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- NAME: string (nullable = true)
 |-- PRCP: double (nullable = true)
 |-- PRCP_ATTRIBUTES: string (nullable = true)
 |-- SNOW: double (nullable = true)
 |-- SNOW_ATTRIBUTES: string (nullable = true)
 |-- SNWD: double (nullable = true)
 |-- SNWD_ATTRIBUTES: string (nullable = true)
 |-- TMAX: double (nullable = true)
 |-- TMAX_ATTRIBUTES: string (nullable = true)
 |-- TMIN: double (nullable = true)
 |-- TMIN_ATTRIBUTES: string (nullable = true)
 |-- AWND: double (nullable = true)
 |-- AWND_ATTRIBUTES: string (nullable = true)
 |-- FMTM: double (nullable = true)
 |-- FMTM_ATTRIBUTES: string (nullable = true)
 |-- PGTM: double (nullable = true)
 |-- PGTM_ATTRIBUTES: string (nullable = true)
 |-- PSUN: double (nullable = true)
 |-- PSUN_ATTRIBUTES: string (nullable = true)
 |-- TAVG: double

In [40]:
#Check Weather Data Quality
# Step 34: Missing values

weather_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in weather_raw.columns
]).show()

# Check duplicate records

print(
    "Duplicate weather rows:",
    weather_raw.count() - weather_raw.dropDuplicates().count()
)

+-------+----+--------+---------+---------+----+----+---------------+-----+---------------+----+---------------+----+---------------+----+---------------+----+---------------+----+---------------+----+---------------+-----+---------------+----+---------------+----+---------------+----+---------------+----+---------------+----+---------------+----+---------------+----+---------------+-----+---------------+-----+---------------+-----+---------------+-----+---------------+-----+---------------+-----+---------------+----+---------------+-----+---------------+-----+---------------+
|STATION|DATE|LATITUDE|LONGITUDE|ELEVATION|NAME|PRCP|PRCP_ATTRIBUTES| SNOW|SNOW_ATTRIBUTES|SNWD|SNWD_ATTRIBUTES|TMAX|TMAX_ATTRIBUTES|TMIN|TMIN_ATTRIBUTES|AWND|AWND_ATTRIBUTES|FMTM|FMTM_ATTRIBUTES|PGTM|PGTM_ATTRIBUTES| PSUN|PSUN_ATTRIBUTES|TAVG|TAVG_ATTRIBUTES|TSUN|TSUN_ATTRIBUTES|WDF2|WDF2_ATTRIBUTES|WDF5|WDF5_ATTRIBUTES|WSF2|WSF2_ATTRIBUTES|WSF5|WSF5_ATTRIBUTES|WT01|WT01_ATTRIBUTES| WT02|WT02_ATTRIBUTES| WT03|WT

In [41]:
#Standardize the Weather Column Names

weather = (
    weather_raw
    .withColumnRenamed("DATE", "weather_date")
    .withColumnRenamed("PRCP", "precipitation")
    .withColumnRenamed("SNOW", "snowfall")
    .withColumnRenamed("SNWD", "snow_depth")
    .withColumnRenamed("TAVG", "avg_temperature")
    .withColumnRenamed("TMAX", "max_temperature")
    .withColumnRenamed("TMIN", "min_temperature")
)

print(weather_raw.columns)

['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'PRCP', 'PRCP_ATTRIBUTES', 'SNOW', 'SNOW_ATTRIBUTES', 'SNWD', 'SNWD_ATTRIBUTES', 'TMAX', 'TMAX_ATTRIBUTES', 'TMIN', 'TMIN_ATTRIBUTES', 'AWND', 'AWND_ATTRIBUTES', 'FMTM', 'FMTM_ATTRIBUTES', 'PGTM', 'PGTM_ATTRIBUTES', 'PSUN', 'PSUN_ATTRIBUTES', 'TAVG', 'TAVG_ATTRIBUTES', 'TSUN', 'TSUN_ATTRIBUTES', 'WDF2', 'WDF2_ATTRIBUTES', 'WDF5', 'WDF5_ATTRIBUTES', 'WSF2', 'WSF2_ATTRIBUTES', 'WSF5', 'WSF5_ATTRIBUTES', 'WT01', 'WT01_ATTRIBUTES', 'WT02', 'WT02_ATTRIBUTES', 'WT03', 'WT03_ATTRIBUTES', 'WT04', 'WT04_ATTRIBUTES', 'WT05', 'WT05_ATTRIBUTES', 'WT06', 'WT06_ATTRIBUTES', 'WT07', 'WT07_ATTRIBUTES', 'WT08', 'WT08_ATTRIBUTES', 'WT09', 'WT09_ATTRIBUTES', 'WT10', 'WT10_ATTRIBUTES']


In [42]:
#Convert Weather Date
weather = weather.withColumn(
    "weather_date",
    F.to_date("weather_date", "yyyy-MM-dd")
)

weather.select("weather_date").show(10)

+------------+
|weather_date|
+------------+
|  1997-05-01|
|  1997-05-02|
|  1997-05-03|
|  1997-05-04|
|  1997-05-05|
|  1997-05-06|
|  1997-05-07|
|  1997-05-08|
|  1997-05-09|
|  1997-05-10|
+------------+
only showing top 10 rows


In [43]:
#Create a Weather Event Indicator
weather = weather.withColumn(
    "rain_event",
    F.when(
        F.col("precipitation") > 0,
        1
    ).otherwise(0)
)
weather.groupBy("rain_event").count().show()

weather = weather.withColumn(
    "snow_event",
    F.when(
        F.col("snowfall") > 0,
        1
    ).otherwise(0)
)
weather.groupBy("snow_event").count().show()


+----------+-----+
|rain_event|count|
+----------+-----+
|         1| 3419|
|         0| 7309|
+----------+-----+

+----------+-----+
|snow_event|count|
+----------+-----+
|         1|   41|
|         0|10687|
+----------+-----+



In [44]:
#Create a Heavy Weather Indicator
weather.select(
    F.min("precipitation").alias("min_precipitation"),
    F.max("precipitation").alias("max_precipitation"),
    F.avg("precipitation").alias("avg_precipitation")
).show()

+-----------------+-----------------+-----------------+
|min_precipitation|max_precipitation|avg_precipitation|
+-----------------+-----------------+-----------------+
|              0.0|           1321.0|24.29553992871592|
+-----------------+-----------------+-----------------+



In [45]:
#Check Weather Date Range
weather.select(
    F.min("weather_date").alias("weather_start"),
    F.max("weather_date").alias("weather_end")
).show()

+-------------+-----------+
|weather_start|weather_end|
+-------------+-----------+
|   1997-05-01| 2026-09-19|
+-------------+-----------+



In [46]:
#Weather Data Quality Summary
weather.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("weather_date").isNull().cast("int")).alias("invalid_dates"),
    F.sum(F.col("precipitation").isNull().cast("int")).alias("missing_precipitation"),
    F.sum(F.col("snowfall").isNull().cast("int")).alias("missing_snowfall")
).show()

+----------+-------------+---------------------+----------------+
|total_rows|invalid_dates|missing_precipitation|missing_snowfall|
+----------+-------------+---------------------+----------------+
|     10728|            0|                  347|           10309|
+----------+-------------+---------------------+----------------+



In [47]:
#Create weather_clean
weather_clean = weather.filter(
    F.col("weather_date").isNotNull()
)

weather_clean = weather_clean.dropDuplicates(["weather_date"])
print("Clean weather rows:", weather_clean.count())

Clean weather rows: 10728


In [48]:
#Cache Weather
weather_clean.cache()

print("Cached weather rows:", weather_clean.count())

Cached weather rows: 10728


In [49]:
#Join Crime with Weather

crime_weather = (
    crime_clean.alias("c")
    .join(
        weather_clean.alias("w"),
        F.col("c.crime_date") == F.col("w.weather_date"),
        "left"
    )
    .select(
        "c.*",
        F.col("w.precipitation"),
        F.col("w.snowfall"),
        F.col("w.avg_temperature"),
        F.col("w.max_temperature"),
        F.col("w.min_temperature"),
        F.col("w.rain_event"),
        F.col("w.snow_event")
    )
)

print("Crime rows:", crime_clean.count())
print("Crime-weather rows:", crime_weather.count())

# Check weather matching

crime_weather.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.col("precipitation").isNotNull().cast("int")
    ).alias("records_with_weather"),
    F.sum(
        F.col("precipitation").isNull().cast("int")
    ).alias("records_without_weather")
).show()

crime_weather.select(
    "ID",
    "crime_date",
    "Primary Type",
    "precipitation",
    "snowfall",
    "avg_temperature",
    "rain_event",
    "snow_event"
).show(10, truncate=False)

Crime rows: 7784664
Crime-weather rows: 7784664
+-------------+--------------------+-----------------------+
|total_records|records_with_weather|records_without_weather|
+-------------+--------------------+-----------------------+
|      7784664|             7769288|                  15376|
+-------------+--------------------+-----------------------+

+--------+----------+------------------+-------------+--------+---------------+----------+----------+
|ID      |crime_date|Primary Type      |precipitation|snowfall|avg_temperature|rain_event|snow_event|
+--------+----------+------------------+-------------+--------+---------------+----------+----------+
|10224738|2015-09-05|BATTERY           |5.0          |NULL    |NULL           |1         |0         |
|10224739|2015-09-04|THEFT             |0.0          |NULL    |NULL           |0         |0         |
|11646166|2018-09-01|THEFT             |10.0         |NULL    |NULL           |1         |0         |
|10224740|2015-09-05|NARCOTICS    

In [50]:
#Join Crime-Weather with Socio-Economic data

socioeconomic_columns = [
    "housing_crowded_pct",
    "poverty_pct",
    "unemployment_pct",
    "no_high_school_pct",
    "dependent_age_pct",
    "per_capita_income",
    "hardship_index"
]

crime_socioeconomic = (
    crime_weather.alias("c")
    .join(
        socioeconomic_clean.alias("s"),
        F.col("c.community_area_clean") == F.col("s.community_area"),
        "left"
    )
    .select(
        "c.*",
        *[
            F.col(f"s.{column}").alias(column)
            for column in socioeconomic_columns
        ]
    )
)

print("Crime-weather rows:", crime_weather.count())
print("Integrated rows:", crime_socioeconomic.count())

crime_socioeconomic.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.col("hardship_index").isNotNull().cast("int")
    ).alias("records_with_socioeconomic_data"),
    F.sum(
        F.col("hardship_index").isNull().cast("int")
    ).alias("records_without_socioeconomic_data")
).show()



Crime-weather rows: 7784664
Integrated rows: 7784664
+-------------+-------------------------------+----------------------------------+
|total_records|records_with_socioeconomic_data|records_without_socioeconomic_data|
+-------------+-------------------------------+----------------------------------+
|      7784664|                        7171112|                            613552|
+-------------+-------------------------------+----------------------------------+



In [51]:
# Create small Community Names lookup

community_names = (
    socioeconomic_clean
    .select(
        F.col("community_area").alias("community_area_key"),
        F.col("community_area_name")
    )
    .dropDuplicates(["community_area_key"])
)

print("Community lookup rows:", community_names.count())

community_names.orderBy("community_area_key").show(100, truncate=False)

Community lookup rows: 77
+------------------+----------------------+
|community_area_key|community_area_name   |
+------------------+----------------------+
|1                 |Rogers Park           |
|2                 |West Ridge            |
|3                 |Uptown                |
|4                 |Lincoln Square        |
|5                 |North Center          |
|6                 |Lake View             |
|7                 |Lincoln Park          |
|8                 |Near North Side       |
|9                 |Edison Park           |
|10                |Norwood Park          |
|11                |Jefferson Park        |
|12                |Forest Glen           |
|13                |North Park            |
|14                |Albany Park           |
|15                |Portage Park          |
|16                |Irving Park           |
|17                |Dunning               |
|18                |Montclaire            |
|19                |Belmont Cragin        |
|20   

In [52]:
from pyspark.sql.functions import broadcast

# Broadcast join: small lookup table + large crime table

integrated_crime = (
    crime_socioeconomic.alias("c")
    .join(
        broadcast(community_names).alias("n"),
        F.col("c.community_area_clean") ==
        F.col("n.community_area_key"),
        "left"
    )
    .drop("community_area_key")
)

print("Final integrated records:", integrated_crime.count())

integrated_crime.explain("formatted")

Final integrated records: 7784664
== Physical Plan ==
AdaptiveSparkPlan (48)
+- Project (47)
   +- BroadcastHashJoin LeftOuter BuildRight (46)
      :- Project (36)
      :  +- BroadcastHashJoin LeftOuter BuildRight (35)
      :     :- Project (30)
      :     :  +- BroadcastHashJoin LeftOuter BuildRight (29)
      :     :     :- InMemoryTableScan (1)
      :     :     :     +- InMemoryRelation (2)
      :     :     :           +- * Project (6)
      :     :     :              +- * Project (5)
      :     :     :                 +- * Filter (4)
      :     :     :                    +- Scan csv  (3)
      :     :     +- BroadcastExchange (28)
      :     :        +- Filter (27)
      :     :           +- InMemoryTableScan (7)
      :     :                 +- InMemoryRelation (8)
      :     :                       +- AdaptiveSparkPlan (26)
                                       +- == Final Plan ==
                                          ResultQueryStage (18)
                         

In [53]:
# Inspect final integrated schema

integrated_crime.printSchema()
integrated_crime.select(
    "ID",
    "Primary Type",
    "crime_timestamp",
    "crime_date",
    "crime_year",
    "crime_month",
    "crime_hour",
    "district_clean",
    "community_area_clean",
    "community_area_name",
    "precipitation",
    "snowfall",
    "avg_temperature",
    "hardship_index"
).show(10, truncate=False)

# Check unique incident IDs

print("Total records:", integrated_crime.count())

print(
    "Unique incident IDs:",
    integrated_crime.select("ID").distinct().count()
)

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- crime_timestamp: timestamp (nullable = true)
 |-- crime_date: date (nullable = true)
 |-- crim

In [54]:
# Define Parquet output path

PROCESSED_PATH = os.path.join(
    BASE_PATH,
    "data",
    "processed"
)

CRIME_PARQUET_PATH = os.path.join(
    PROCESSED_PATH,
    "integrated_crime_parquet"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Parquet output:", CRIME_PARQUET_PATH)

Parquet output: C:\Users\HP\Documents\Big Data Technology\data\processed\integrated_crime_parquet


In [55]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["HADOOP_HOME_DIR"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("HADOOP_HOME_DIR:", os.environ["HADOOP_HOME_DIR"])
print("winutils:", os.path.isfile(r"C:\hadoop\bin\winutils.exe"))

HADOOP_HOME: C:\hadoop
HADOOP_HOME_DIR: C:\hadoop
winutils: True


In [56]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ChicagoCrimeBigData")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config(
        "spark.hadoop.fs.file.impl",
        "org.apache.hadoop.fs.RawLocalFileSystem"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print(
    "Hadoop version:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Spark version: 4.2.0
Hadoop version: 3.5.0


In [57]:
# Write partitioned Parquet dataset

(
    integrated_crime
    .write
    .mode("overwrite")
    .format("parquet")
    .partitionBy("crime_year", "district_clean")
    .save(CRIME_PARQUET_PATH)
)

print("Integrated crime dataset saved successfully.")

Integrated crime dataset saved successfully.


In [58]:
crime_parquet = spark.read.parquet(CRIME_PARQUET_PATH)

print("Parquet rows:", crime_parquet.count())
print("Parquet columns:", len(crime_parquet.columns))
crime_parquet.printSchema()

Parquet rows: 7784664
Parquet columns: 48
root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- crime_timestamp: timestamp (nullable = true)
 |-- cr

In [59]:
# Compare counts

source_count = integrated_crime.count()
parquet_count = crime_parquet.count()

print("Source count:", source_count)
print("Parquet count:", parquet_count)
print("Counts match:", source_count == parquet_count)

Source count: 7784664
Parquet count: 7784664
Counts match: True


In [60]:
print("Integrated crime rows:", integrated_crime.count())

print("\nIntegrated crime columns:")
print(integrated_crime.columns)

print("\nSchema:")
integrated_crime.printSchema()

Integrated crime rows: 7784664

Integrated crime columns:
['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location', 'crime_timestamp', 'crime_date', 'crime_year', 'crime_month', 'crime_day', 'crime_hour', 'crime_day_of_week', 'district_clean', 'community_area_clean', 'latitude_clean', 'longitude_clean', 'precipitation', 'snowfall', 'avg_temperature', 'max_temperature', 'min_temperature', 'rain_event', 'snow_event', 'housing_crowded_pct', 'poverty_pct', 'unemployment_pct', 'no_high_school_pct', 'dependent_age_pct', 'per_capita_income', 'hardship_index', 'community_area_name']

Schema:
root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |

In [61]:
# Filter by year and district

sample_year = 2020
sample_district = 1

filtered_crime = crime_parquet.filter(
    (F.col("crime_year") == sample_year) &
    (F.col("district_clean") == sample_district)
)

filtered_crime.explain("formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [48]: [ID#35909, Case Number#35910, Date#35911, Block#35912, IUCR#35913, Primary Type#35914, Description#35915, Location Description#35916, Arrest#35917, Domestic#35918, Beat#35919, District#35920, Ward#35921, Community Area#35922, FBI Code#35923, X Coordinate#35924, Y Coordinate#35925, Year#35926, Updated On#35927, Latitude#35928, Longitude#35929, Location#35930, crime_timestamp#35931, crime_date#35932, crime_month#35933, crime_day#35934, crime_hour#35935, crime_day_of_week#35936, community_area_clean#35937, latitude_clean#35938, longitude_clean#35939, precipitation#35940, snowfall#35941, avg_temperature#35942, max_temperature#35943, min_temperature#35944, rain_event#35945, snow_event#35946, housing_crowded_pct#35947, poverty_pct#35948, unemployment_pct#35949, no_high_school_pct#35950, dependent_age_pct#35951, per_capita_income#35952, hardship_index#35953, community_area_name#35954, crime_year#3595

In [62]:
filtered_crime.select(
    "ID",
    "Primary Type",
    "crime_year",
    "district_clean",
    "crime_date"
).show(10, truncate=False)

+--------+------------------+----------+--------------+----------+
|ID      |Primary Type      |crime_year|district_clean|crime_date|
+--------+------------------+----------+--------------+----------+
|12640536|ASSAULT           |2020      |1             |2020-03-05|
|12042062|THEFT             |2020      |1             |2020-04-24|
|12498699|DECEPTIVE PRACTICE|2020      |1             |2020-06-01|
|12252209|OTHER OFFENSE     |2020      |1             |2020-08-01|
|12302029|THEFT             |2020      |1             |2020-11-20|
|12038085|BATTERY           |2020      |1             |2020-04-26|
|12571403|DECEPTIVE PRACTICE|2020      |1             |2020-11-10|
|12249150|THEFT             |2020      |1             |2020-12-19|
|12571480|DECEPTIVE PRACTICE|2020      |1             |2020-12-05|
|12038238|OTHER OFFENSE     |2020      |1             |2020-04-26|
+--------+------------------+----------+--------------+----------+
only showing top 10 rows


In [63]:
# Save clean source datasets as Parquet

SOCIO_PARQUET_PATH = os.path.join(
    PROCESSED_PATH,
    "socioeconomic_parquet"
)

WEATHER_PARQUET_PATH = os.path.join(
    PROCESSED_PATH,
    "weather_parquet"
)

socioeconomic_clean.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(SOCIO_PARQUET_PATH)

weather_clean.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(WEATHER_PARQUET_PATH)

print("Clean socio-economic and weather datasets saved.")

Clean socio-economic and weather datasets saved.


In [64]:
# Final ETL summary

print("========== TASK 1 ETL SUMMARY ==========")

print("Crime source rows:", crime_clean.count())
print("Socio-economic rows:", socioeconomic_clean.count())
print("Weather rows:", weather_clean.count())

print("Integrated crime rows:", integrated_crime.count())
print("Saved Parquet rows:", crime_parquet.count())

print(
    "Integrated and Parquet counts match:",
    integrated_crime.count() == crime_parquet.count()
)

print(
    "Crime partition columns:",
    ["crime_year", "district_clean"]
)

print("========================================")

========== TASK 1 ETL SUMMARY ==========
Crime source rows: 7784664
Socio-economic rows: 77
Weather rows: 10728
Integrated crime rows: 7784664
Saved Parquet rows: 7784664
Integrated and Parquet counts match: True
Crime partition columns: ['crime_year', 'district_clean']
